# Optimizacion de la silueta de los perfiles de una pala para reduccion de ruido

**Funcion objetivo: el OASPL(A) de la pala completa**, no el de una seccion aislada. La
cadena de evaluacion es la del esquema de trabajo:

```
X1, X2  ->  parametrizacion  ->  XFOIL          ->  OpenFAST          ->  AMIET        ->  OASPL_pala
(secciones  (CST / PARSEC)      (polares +/-180   (Cp y condiciones     (ruido de borde   (suma sobre
 de control)                     de cada seccion)  locales por estacion) de fuga por tira)  la envergadura)
                                                                                                |
                    <------------------  dual annealing  <-------------------------------------
```

Las variables de diseno son los coeficientes de las **secciones de control** (X1 = NACA
63-221, X2 = NACA 63-218); el resto de estaciones radiales de la pala se obtiene
interpolando entre ellas. Cada evaluacion del objetivo recorre la cadena entera y devuelve
un unico numero: el OASPL(A) radiado por la pala. El coeficiente de potencia `Cp` que
devuelve OpenFAST entra como restriccion, para que la reduccion de ruido no se pague con
perdida de rendimiento.

**Flujo del libro**

| Seccion | Contenido | Estado |
|---|---|---|
| 1 | Configuracion global: fluido, rotor y discretizacion de la pala | operativa |
| 2 | Lectura de las secciones de control y construccion de la geometria de la pala | operativa |
| 3 | Caracterizacion aerodinamica con XFOIL: polares +/-180 de cada seccion | operativa |
| 4 | Analisis aeroelastico de la pala en OpenFAST: Cp y condiciones locales | **plantilla a completar** |
| 5 | Ruido de borde de fuga (Amiet) por tiras e integracion sobre la pala | **plantilla a completar** (la suma sobre la pala si esta escrita) |
| 6 | Parametrizacion de las siluetas: CST (Kulfan) y PARSEC | operativa |
| 7 | Optimizacion con Dual Annealing sobre el OASPL de la pala | operativa |

Las secciones 4 y 5 contienen la firma y el **contrato de datos** cerrado de
`calcular_respuesta_openfast` y `calcular_ruido_amiet`. El resto del libro ya esta escrito
contra ese contrato: al implementarlas, la cadena queda cerrada sin tocar nada mas.

## 1. Configuracion global: fluido, rotor y discretizacion de la pala

Tres bloques de datos, separados porque cambian por motivos distintos:

- `COND` — fluido y condiciones de referencia comunes (temperatura, viscosidad, densidad,
  transicion forzada, intensidad de turbulencia y posicion del observador). De `Tu` sale el
  `N_crit` del metodo e^N de XFOIL, con la misma correlacion que el codigo de ruido:
  `N_crit = -8.43 - 2.4*ln(Tu/100)`.
- `PALA` — geometria y discretizacion radial: radio, numero de palas, la tabla de
  estaciones (r, cuerda, torsion) y la posicion radial de las **secciones de control** que
  parametriza el optimizador.
- `ROTOR` — punto de operacion: velocidad de viento, velocidad de giro y paso.

Reynolds y Mach ya no son globales: cada estacion radial tiene los suyos, calculados con su
cuerda y su velocidad relativa, que es precisamente lo que devuelve OpenFAST (seccion 4).

> La tabla de pala que viene por defecto es **provisional** (distribucion suave de cuerda y
> torsion, marcada con `PALA_PROVISIONAL = True`). Sustituyela por la del fichero de pala de
> AeroDyn de tu modelo en cuanto lo tengas: es el unico dato de esta celda que no procede
> del caso de ruido ya validado.

In [ ]:
"""Configuracion global: fluido, rotor, discretizacion de la pala y rutas."""
from __future__ import annotations

import re
import subprocess
import tempfile
import time
from dataclasses import dataclass, field, replace
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator
from scipy.optimize import dual_annealing
from scipy.special import comb

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

# --------------------------------------------------------------------------------------
# Rutas del proyecto
# --------------------------------------------------------------------------------------
RAIZ = Path.cwd()
SALIDA = RAIZ / "resultados_notebook"
SALIDA.mkdir(exist_ok=True)


def localizar_xfoil(raiz: Path) -> Path:
    """Devuelve la ruta a xfoil.exe buscando primero en las ubicaciones conocidas."""
    candidatos = [
        raiz / "xfoil.exe",
        raiz / "work" / "Amiet-Theory-for-TE-Noise" / "xfoil.exe",
        raiz / "entrega_codigo_limpio" / "xfoil.exe",
    ]
    for c in candidatos:
        if c.is_file():
            return c.resolve()
    hallados = sorted(raiz.glob("**/xfoil.exe"))
    if hallados:
        return hallados[0].resolve()
    raise FileNotFoundError("No se encontro xfoil.exe: asigne XFOIL_EXE manualmente.")


XFOIL_EXE = localizar_xfoil(RAIZ)

# Secciones de control: los X1, X2 que parametriza el optimizador. Ficheros de dos columnas
# x/c, y/c (admiten cabecera con el nombre). "r_R" es su posicion radial en la pala.
PERFILES = RAIZ / "work" / "Amiet-Theory-for-TE-Noise"
SECCIONES_CONTROL = [
    {"archivo": PERFILES / "NACA63221.txt", "r_R": 0.40, "etiqueta": "X1 (raiz/medio)"},
    {"archivo": PERFILES / "NACA63218.txt", "r_R": 0.95, "etiqueta": "X2 (punta)"},
]


# --------------------------------------------------------------------------------------
# Fluido y condiciones de referencia
# --------------------------------------------------------------------------------------
@dataclass
class CondicionesOperacion:
    """Fluido, capa limite y observador. Cuerda/velocidad/AoA se fijan por estacion."""

    # valores de referencia de una seccion (se sobreescriben por estacion con `replace`)
    cuerda: float = 0.57            # m
    envergadura: float = 1.0        # m, ancho de la tira
    U: float = 83.31                # m/s, velocidad relativa de la seccion
    AoA: float = 4.0                # deg
    # capa limite
    xtr_s: float = 0.05             # x/c de transicion forzada, extrados (succion)
    xtr_p: float = 0.05             # x/c de transicion forzada, intrados (presion)
    Tu: float = 0.07                # % de intensidad de turbulencia -> N_crit
    x_bl: float = 0.99              # x/c donde se extraen los parametros de capa limite
    # observador acustico, en m (sistema de la torre: x aguas abajo, z vertical)
    obs_x: float = 0.0
    obs_y: float = 0.0
    obs_z: float = 1.0
    # fluido
    T_C: float = 22.0               # degC
    nu: float = 1.5e-5              # m^2/s, viscosidad cinematica
    gamma: float = 1.4
    R_gas: float = 287.0            # J/(kg K)
    rho: float = 1.181              # kg/m^3

    @property
    def T_K(self) -> float:
        return self.T_C + 273.15

    @property
    def c0(self) -> float:
        """Velocidad del sonido [m/s]."""
        return float(np.sqrt(self.gamma * self.R_gas * self.T_K))

    @property
    def Mach(self) -> float:
        return self.U / self.c0

    @property
    def Re(self) -> float:
        return self.cuerda * self.U / self.nu

    @property
    def Ncrit(self) -> float:
        """N_crit del metodo e^N a partir de la intensidad de turbulencia."""
        return -8.43 - 2.4 * float(np.log(self.Tu / 100.0))


COND = CondicionesOperacion()


# --------------------------------------------------------------------------------------
# Rotor y pala
# --------------------------------------------------------------------------------------
@dataclass
class Rotor:
    """Punto de operacion del rotor."""

    U_inf: float = 10.0             # m/s, viento incidente
    Omega: float = 12.1             # rpm
    paso: float = 0.0               # deg, paso colectivo
    n_palas: int = 3

    @property
    def Omega_rad(self) -> float:
        return self.Omega * 2.0 * np.pi / 60.0


@dataclass
class Pala:
    """Geometria y discretizacion radial de la pala."""

    radio: float = 63.0             # m, radio del rotor
    r_raiz: float = 0.10            # r/R donde empieza la parte aerodinamica
    n_estaciones: int = 16          # numero de tiras
    provisional: bool = True        # True mientras la tabla no venga del modelo AeroDyn
    r_R: np.ndarray = field(default=None)
    cuerda: np.ndarray = field(default=None)   # m
    torsion: np.ndarray = field(default=None)  # deg

    def __post_init__(self):
        if self.r_R is None:
            self.r_R = np.linspace(self.r_raiz, 1.0, self.n_estaciones)
        # TABLA PROVISIONAL: distribuciones suaves tipo pala de gran aerogenerador.
        # Sustituir por la tabla del fichero de pala de AeroDyn cuando este disponible.
        if self.cuerda is None:
            self.cuerda = self.radio * (0.115 - 0.085 * (self.r_R - 0.1) / 0.9)
        if self.torsion is None:
            self.torsion = 13.0 * np.exp(-3.0 * (self.r_R - 0.1))

    @property
    def r(self) -> np.ndarray:
        """Posicion radial de cada estacion [m]."""
        return self.r_R * self.radio

    @property
    def dr(self) -> np.ndarray:
        """Ancho radial de cada tira [m], para la suma de ruido sobre la envergadura."""
        bordes = np.empty(len(self.r) + 1)
        bordes[1:-1] = 0.5 * (self.r[1:] + self.r[:-1])
        bordes[0] = self.r[0] - (bordes[1] - self.r[0])
        bordes[-1] = self.r[-1] + (self.r[-1] - bordes[-2])
        return np.diff(bordes)


ROTOR = Rotor()
PALA = Pala()
PALA_PROVISIONAL = PALA.provisional

print(f"Raiz del proyecto  : {RAIZ}")
print(f"XFOIL              : {XFOIL_EXE}")
print(f"Carpeta de salida  : {SALIDA}")

print("\nSecciones de control (variables de diseno del optimizador)")
print("-" * 68)
for s in SECCIONES_CONTROL:
    marca = "OK " if s["archivo"].is_file() else "NO "
    print(f"  [{marca}] r/R = {s['r_R']:.2f}  {s['etiqueta']:<18} {s['archivo'].name}")

print("\nFluido y observador")
print("-" * 68)
print(f"  T = {COND.T_C:.1f} degC | rho = {COND.rho:.3f} kg/m3 | c0 = {COND.c0:.2f} m/s")
print(f"  nu = {COND.nu:.2e} m2/s | Ncrit = {COND.Ncrit:.3f} (Tu = {COND.Tu} %)")
print(f"  transicion forzada x/c = {COND.xtr_s:.2f} / {COND.xtr_p:.2f} | "
      f"capa limite en x/c = {COND.x_bl}")
print(f"  observador (x, y, z) = ({COND.obs_x}, {COND.obs_y}, {COND.obs_z}) m")

print("\nRotor y pala")
print("-" * 68)
print(f"  U_inf = {ROTOR.U_inf:.2f} m/s | Omega = {ROTOR.Omega:.2f} rpm "
      f"({ROTOR.Omega_rad:.3f} rad/s) | paso = {ROTOR.paso:.1f} deg | "
      f"{ROTOR.n_palas} palas")
print(f"  radio = {PALA.radio:.2f} m | {PALA.n_estaciones} estaciones "
      f"en r/R = [{PALA.r_R[0]:.2f}, {PALA.r_R[-1]:.2f}]")
print(f"  velocidad de punta = {ROTOR.Omega_rad * PALA.radio:.1f} m/s | "
      f"lambda = {ROTOR.Omega_rad * PALA.radio / ROTOR.U_inf:.2f}")
if PALA_PROVISIONAL:
    print("\n  AVISO: la tabla de cuerda y torsion es PROVISIONAL. Sustituyela por la")
    print("         del fichero de pala de AeroDyn en cuanto tengas el modelo OpenFAST.")

## 2. Lectura de las secciones de control y construccion de la geometria de la pala

Se leen los ficheros de dos columnas (`x/c`, `y/c`) en orden Selig/XFOIL: el contorno
arranca en el borde de fuga, recorre el extrados hasta el borde de ataque y vuelve por el
intrados. Para cada seccion de control la rutina descarta cabeceras, normaliza por la
cuerda, separa extrados e intrados en el punto de minima `x` y calcula sus propiedades
geometricas (espesor y curvatura maximos, radio de borde de ataque, apertura del borde de
fuga).

Despues se construye la **geometria de la pala**: cada estacion radial recibe un perfil
interpolado linealmente entre las secciones de control vecinas, sobre una malla `x/c` comun
de espaciado coseno. Es la practica habitual en definicion de palas y es la que hace que
mover un coeficiente de `X1` o `X2` modifique la pala entera de forma continua, no a
saltos.

In [ ]:
"""Lectura de las secciones de control, interpolacion radial y visualizacion."""

N_MALLA = 121          # puntos por cara de la malla comun (espaciado coseno)
X_MALLA = 0.5 * (1.0 - np.cos(np.linspace(0.0, np.pi, N_MALLA)))


def _leer_dos_columnas(ruta: Path) -> np.ndarray:
    """Extrae las filas numericas (x, y) de un fichero de coordenadas."""
    filas = []
    for linea in Path(ruta).read_text(errors="ignore").splitlines():
        partes = linea.replace(",", " ").split()
        if len(partes) < 2:
            continue
        try:
            filas.append([float(partes[0]), float(partes[1])])
        except ValueError:
            continue  # cabecera con el nombre del perfil u otro texto
    if len(filas) < 10:
        raise ValueError(f"{ruta} no contiene suficientes pares de coordenadas.")
    return np.asarray(filas, dtype=float)


def cargar_perfil(ruta, normalizar: bool = True) -> dict:
    """Lee un perfil Selig/XFOIL y devuelve el contorno y las dos superficies.

    Claves devueltas:
      nombre     nombre del perfil (nombre del fichero sin extension)
      x, y       contorno completo (BF -> extrados -> BA -> intrados -> BF)
      xu, yu     extrados, con x creciente desde el borde de ataque
      xl, yl     intrados, con x creciente desde el borde de ataque
      geom       diccionario con las propiedades geometricas
    """
    ruta = Path(ruta)
    xy = _leer_dos_columnas(ruta)

    # Algunos exportadores anteponen un punto de referencia (p.ej. 0.5, 0) al contorno.
    if len(xy) >= 3 and 0.05 < xy[0, 0] < 0.95 and abs(xy[1, 0] - 1.0) < 1e-3:
        xy = xy[1:]

    x, y = xy[:, 0].copy(), xy[:, 1].copy()

    if normalizar:
        cuerda_geom = x.max() - x.min()
        if abs(cuerda_geom - 1.0) > 1e-6 or abs(x.min()) > 1e-6:
            y = y / cuerda_geom
            x = (x - x.min()) / cuerda_geom

    i_ba = int(np.argmin(x))                            # borde de ataque = x minima
    xu, yu = x[: i_ba + 1][::-1], y[: i_ba + 1][::-1]    # extrados, x creciente
    xl, yl = x[i_ba:], y[i_ba:]                         # intrados, x creciente

    perfil = {
        "nombre": ruta.stem, "ruta": ruta,
        "x": x, "y": y, "xu": xu, "yu": yu, "xl": xl, "yl": yl,
    }
    perfil["geom"] = propiedades_geometricas(perfil)
    return perfil


def propiedades_geometricas(perfil: dict) -> dict:
    """Espesor, curvatura, radio de borde de ataque y apertura del borde de fuga."""
    xs = X_MALLA
    yu = PchipInterpolator(perfil["xu"], perfil["yu"])(xs)
    yl = PchipInterpolator(perfil["xl"], perfil["yl"])(xs)

    espesor = yu - yl
    curvatura = 0.5 * (yu + yl)
    i_t = int(np.argmax(espesor))
    i_c = int(np.argmax(np.abs(curvatura)))

    # Radio de borde de ataque: ajuste y ~ a*sqrt(x) en el extrados cerca del BA.
    cerca = (perfil["xu"] > 0) & (perfil["xu"] < 0.05)
    a1 = float(np.linalg.lstsq(np.sqrt(perfil["xu"][cerca])[:, None],
                               perfil["yu"][cerca], rcond=None)[0][0])

    return {
        "x_malla": xs, "yu_malla": yu, "yl_malla": yl,
        "espesor": espesor, "curvatura": curvatura,
        "t_max": float(espesor[i_t]), "x_t_max": float(xs[i_t]), "i_t": i_t,
        "camber_max": float(curvatura[i_c]), "x_camber_max": float(xs[i_c]),
        "r_le": float(a1 ** 2 / 2.0),
        "gap_te": float(perfil["yu"][-1] - perfil["yl"][-1]),
        "n_puntos": int(len(perfil["x"])),
    }


def perfil_desde_caras(nombre: str, xs, yu, yl) -> dict:
    """Ensambla un perfil (contorno Selig + caras) a partir de extrados e intrados."""
    perfil = {
        "nombre": nombre, "ruta": None,
        "x": np.concatenate([xs[::-1], xs[1:]]),
        "y": np.concatenate([np.asarray(yu)[::-1], np.asarray(yl)[1:]]),
        "xu": np.asarray(xs), "yu": np.asarray(yu),
        "xl": np.asarray(xs), "yl": np.asarray(yl),
    }
    perfil["geom"] = propiedades_geometricas(perfil)
    return perfil


def interpolar_perfil(secciones: list, r_R: float) -> dict:
    """Perfil de una estacion radial, interpolado entre las secciones de control.

    Fuera del intervalo cubierto por las secciones de control se mantiene la seccion
    extrema (no se extrapola la forma, que es lo habitual en definicion de palas).
    """
    posiciones = np.array([s["r_R"] for s in secciones], float)
    orden = np.argsort(posiciones)
    posiciones, secciones = posiciones[orden], [secciones[i] for i in orden]

    if r_R <= posiciones[0]:
        base = secciones[0]["perfil"]
        return perfil_desde_caras(base["nombre"], X_MALLA,
                                  base["geom"]["yu_malla"], base["geom"]["yl_malla"])
    if r_R >= posiciones[-1]:
        base = secciones[-1]["perfil"]
        return perfil_desde_caras(base["nombre"], X_MALLA,
                                  base["geom"]["yu_malla"], base["geom"]["yl_malla"])

    j = int(np.searchsorted(posiciones, r_R)) - 1
    w = (r_R - posiciones[j]) / (posiciones[j + 1] - posiciones[j])
    g0 = secciones[j]["perfil"]["geom"]
    g1 = secciones[j + 1]["perfil"]["geom"]
    yu = (1 - w) * g0["yu_malla"] + w * g1["yu_malla"]
    yl = (1 - w) * g0["yl_malla"] + w * g1["yl_malla"]
    nombre = (f"{secciones[j]['perfil']['nombre']}-"
              f"{secciones[j + 1]['perfil']['nombre']}_{w:.2f}")
    return perfil_desde_caras(nombre, X_MALLA, yu, yl)


def construir_pala(secciones: list, pala: Pala) -> list:
    """Geometria de cada estacion radial: perfil interpolado, cuerda y torsion."""
    estaciones = []
    for k, r_R in enumerate(pala.r_R):
        p = interpolar_perfil(secciones, float(r_R))
        estaciones.append({
            "k": k, "r_R": float(r_R), "r": float(pala.r[k]),
            "dr": float(pala.dr[k]), "cuerda": float(pala.cuerda[k]),
            "torsion": float(pala.torsion[k]), "perfil": p,
            "t_c": p["geom"]["t_max"],
        })
    return estaciones


def dibujar_pala(secciones: list, estaciones: list):
    """Siluetas de las secciones de control y distribuciones radiales de la pala."""
    fig = plt.figure(figsize=(11.0, 6.6))
    ejes = [fig.add_subplot(2, 2, 1), fig.add_subplot(2, 2, 2),
            fig.add_subplot(2, 2, 3), fig.add_subplot(2, 2, 4)]

    ax = ejes[0]
    for s, color in zip(secciones, plt.cm.viridis(np.linspace(0, 0.8, len(secciones)))):
        p = s["perfil"]
        g = p["geom"]
        ax.plot(p["x"], p["y"], "-", color=color, lw=1.4,
                label=f"{s['etiqueta']}: {p['nombre']} ({100 * g['t_max']:.1f}% c)")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-0.02, 1.02)
    ax.set_xlabel("x/c"); ax.set_ylabel("y/c")
    ax.legend(fontsize=7, loc="lower center")
    ax.set_title("(a) Secciones de control")

    ax = ejes[1]
    for e in estaciones:
        col = plt.cm.viridis(e["r_R"])
        ax.plot(e["perfil"]["x"], e["perfil"]["y"] + 0.0, "-", color=col, lw=0.8)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-0.02, 1.02)
    ax.set_xlabel("x/c"); ax.set_ylabel("y/c")
    ax.set_title(f"(b) Perfiles interpolados en las {len(estaciones)} estaciones")

    ax = ejes[2]
    ax.plot([e["r_R"] for e in estaciones], [e["cuerda"] for e in estaciones], "-o",
            color="tab:blue", ms=3)
    ax.set_xlabel("r/R"); ax.set_ylabel("cuerda [m]")
    ax.set_title("(c) Distribucion de cuerda")

    ax = ejes[3]
    ax.plot([e["r_R"] for e in estaciones], [e["torsion"] for e in estaciones], "-o",
            color="tab:red", ms=3, label="torsion")
    ax.set_xlabel("r/R"); ax.set_ylabel("torsion [deg]")
    ax2 = ax.twinx()
    ax2.plot([e["r_R"] for e in estaciones], [100 * e["t_c"] for e in estaciones], "--s",
             color="0.4", ms=3, label="espesor")
    ax2.set_ylabel("espesor [% c]")
    ax2.grid(False)
    ax.set_title("(d) Torsion y espesor relativo")

    fig.tight_layout()
    return fig


# --------------------------------------------------------------------------------------
for s in SECCIONES_CONTROL:
    s["perfil"] = cargar_perfil(s["archivo"])

ESTACIONES = construir_pala(SECCIONES_CONTROL, PALA)

print("Secciones de control")
print("-" * 76)
print(f"{'etiqueta':<20}{'perfil':<14}{'r/R':>6}{'pts':>6}{'t/c':>9}{'camber':>9}{'r_LE':>9}")
for s in SECCIONES_CONTROL:
    g = s["perfil"]["geom"]
    print(f"{s['etiqueta']:<20}{s['perfil']['nombre']:<14}{s['r_R']:>6.2f}"
          f"{g['n_puntos']:>6d}{100 * g['t_max']:>8.2f}%{100 * g['camber_max']:>8.2f}%"
          f"{g['r_le']:>9.5f}")

print(f"\nPala discretizada en {len(ESTACIONES)} tiras")
print("-" * 76)
print(f"{'k':>3}{'r/R':>7}{'r [m]':>9}{'dr [m]':>9}{'cuerda [m]':>12}"
      f"{'torsion [deg]':>15}{'t/c [%]':>10}")
for e in ESTACIONES:
    print(f"{e['k']:>3d}{e['r_R']:>7.3f}{e['r']:>9.2f}{e['dr']:>9.2f}"
          f"{e['cuerda']:>12.3f}{e['torsion']:>15.2f}{100 * e['t_c']:>10.2f}")

fig = dibujar_pala(SECCIONES_CONTROL, ESTACIONES)
fig.savefig(SALIDA / "fig_geometria_pala.png", bbox_inches="tight")
plt.show()

## 3. Caracterizacion aerodinamica con XFOIL: polares +/-180 de cada seccion

Se ejecuta `xfoil.exe` sobre las coordenadas **exactas** de cada seccion (se escriben tal
cual en un `.dat` sin cabecera, el formato que XFOIL lee sin ambiguedad), cada una con su
propio Reynolds y Mach, calculados con su cuerda y su velocidad relativa. La transicion se
fuerza en `x/c = 0.05` en ambas caras y el `N_crit` sale de `Tu`, igual que en el caso de
ruido ya validado.

Dos funciones cubren los dos usos distintos que tiene XFOIL en esta cadena:

- `calcular_aerodinamica_xfoil` — barrido completo de una seccion: punto de referencia,
  polar convergida (dos marchas desde 0 deg, hacia +25 y hacia -20 deg) y **polar extendida
  a +/-180 deg** por el metodo de **Viterna** (estilo AirfoilPrep, `CDmax = 1.3`,
  `cl_adj = 0.7`) a paso de 1 deg. Esa polar +/-180 es el insumo que necesita AeroDyn
  (seccion 4): XFOIL solo converge en el rango adherido y fuera de el hay que extrapolar.
- `calcular_capa_limite_xfoil` — una sola corrida a un angulo de ataque dado, que devuelve
  `Cl`, `Cd`, `Cm` y los parametros de capa limite en `x/c = 0.99` de ambas caras
  (`delta*`, `theta`, `Cf`, `Ue`, `H`). Es la que se llama **por estacion radial** con el
  angulo local que devuelve OpenFAST, porque esos son exactamente los datos que consume el
  modelo de ruido de la seccion 5.

> Detalle practico: pasado el angulo de perdida XFOIL puede quedarse iterando
> indefinidamente sobre NaN (`MRCHDU`/`TRCHEK2` sin convergencia). Como los puntos
> convergidos se escriben al fichero de polar segun se obtienen, la corrida se vigila y se
> corta en cuanto ese fichero deja de crecer, conservando todo lo convergido. Sin esa
> vigilancia un barrido tarda minutos en lugar de segundos.

In [ ]:
"""XFOIL: polares +/-180 de cada seccion y capa limite en el angulo local de cada tira."""

# Pasado el angulo de perdida, XFOIL puede quedarse iterando sobre NaN indefinidamente
# (MRCHDU/TRCHEK2 sin convergencia). Los puntos ya convergidos se escriben al fichero de
# polar conforme se obtienen, de modo que se vigila ese fichero: si deja de crecer durante
# ESPERA_SIN_AVANCE segundos se mata el proceso y se conserva lo convergido hasta ahi.
TIEMPO_MAX_XFOIL = 180      # s, tope absoluto de una corrida
TIEMPO_MAX_PUNTO = 20       # s, tope de una corrida a angulo de ataque fijo
ESPERA_SIN_AVANCE = 6.0     # s sin puntos nuevos -> se considera atascado


def escribir_dat(ruta: Path, x, y) -> Path:
    """Escribe las coordenadas en el formato Selig de dos columnas sin cabecera."""
    ruta = Path(ruta)
    with ruta.open("w") as f:
        for xi, yi in zip(np.asarray(x).ravel(), np.asarray(y).ravel()):
            f.write(f"{xi:9.5f} {yi:9.5f}\n")
    return ruta


def _ejecutar_xfoil(comandos: str, carpeta: Path, tiempo_max: float = TIEMPO_MAX_XFOIL,
                    fichero_vigilado: str | None = None,
                    espera_sin_avance: float = ESPERA_SIN_AVANCE) -> str:
    """Lanza xfoil.exe con la secuencia de comandos y devuelve su salida de texto.

    Si se indica `fichero_vigilado` (el fichero PACC del barrido), se mata el proceso
    cuando deja de crecer: es la señal de que XFOIL se ha atascado iterando sobre NaN.
    """
    kwargs = {}
    if hasattr(subprocess, "STARTUPINFO"):          # Windows: sin ventana emergente
        si = subprocess.STARTUPINFO()
        si.dwFlags |= subprocess.STARTF_USESHOWWINDOW
        kwargs["startupinfo"] = si

    ruta_salida = carpeta / "xfoil_stdout.txt"
    vigilado = (carpeta / fichero_vigilado) if fichero_vigilado else None

    with ruta_salida.open("w") as fsal:
        proc = subprocess.Popen(
            [str(XFOIL_EXE)], stdin=subprocess.PIPE, stdout=fsal,
            stderr=subprocess.STDOUT, text=True, cwd=str(carpeta), **kwargs,
        )
        try:
            proc.stdin.write(comandos)
            proc.stdin.flush()
            proc.stdin.close()
        except OSError:
            pass

        t0 = ultimo_avance = time.time()
        tam = -1
        while proc.poll() is None:
            time.sleep(0.2)
            ahora = time.time()
            if vigilado is not None:
                nuevo = vigilado.stat().st_size if vigilado.is_file() else -1
                if nuevo != tam:
                    tam, ultimo_avance = nuevo, ahora
                if nuevo >= 0 and ahora - ultimo_avance > espera_sin_avance:
                    proc.kill()                     # atascado: no salen puntos nuevos
                    break
            if ahora - t0 > tiempo_max:
                proc.kill()
                break
        proc.wait()

    # La salida se lee acotada: un XFOIL atascado escribe megabytes de avisos de NaN.
    with ruta_salida.open(errors="ignore") as f:
        return f.read(500_000)


def _preambulo_xfoil(fichero_dat: str, cond: CondicionesOperacion, iteraciones: int) -> str:
    """Carga del perfil y ajuste del solver viscoso (Re, Mach, N_crit, transicion)."""
    return (
        "PLOP\nG F\n\n"
        f"load\n{fichero_dat}\nnew\n"
        "pane\n"
        "\n\noper\n"
        f"visc\n{cond.Re:g}\n"
        f"mach {cond.Mach:g}\n"
        f"iter {iteraciones:d}\n"
        f"vpar\nN {cond.Ncrit:g}\n\n"
        f"vpar\nxtr\n{cond.xtr_s:g}\n{cond.xtr_p:g}\n\n"
    )


def _leer_polar(ruta: Path) -> np.ndarray:
    """Lee un fichero PACC de XFOIL -> matriz [alpha, Cl, Cd, Cdp, Cm]."""
    if not Path(ruta).is_file():
        return np.zeros((0, 5))
    filas = []
    for linea in Path(ruta).read_text(errors="ignore").splitlines():
        partes = linea.split()
        if len(partes) < 5:
            continue
        try:
            filas.append([float(v) for v in partes[:5]])
        except ValueError:
            continue                                # lineas de cabecera
    return np.asarray(filas, dtype=float) if filas else np.zeros((0, 5))


def _leer_dump(ruta: Path, salida_xfoil: str, x_bl: float, cuerda: float) -> dict:
    """Parametros de capa limite en x/c = x_bl a partir del DUMP de XFOIL.

    Columnas del DUMP viscoso: s, x, y, Ue/Vinf, delta*, theta, Cf, H.
    El recorrido es BF -> extrados -> BA -> intrados -> BF y despues la estela, por lo
    que el punto homologo del intrados es el simetrico respecto al numero de nodos.
    """
    if not Path(ruta).is_file():
        return {}
    datos = []
    for linea in Path(ruta).read_text(errors="ignore").splitlines()[1:]:
        partes = linea.split()
        if len(partes) < 8:
            continue
        try:
            datos.append([float(v) for v in partes[:8]])
        except ValueError:
            continue
    if not datos:
        return {}
    D = np.asarray(datos)

    m = re.search(r"Number of panel nodes\D*(\d+)", salida_xfoil)
    if m:
        n_nodos = int(m.group(1))
    else:                                            # respaldo: la estela tiene x > 1
        n_nodos = int(np.max(np.nonzero(D[:, 1] <= 1.0 + 1e-6)[0])) + 1
    n_nodos = min(n_nodos, len(D))

    x_c = D[:, 1]
    i_s = int(np.argmin(np.abs(x_bl - x_c[: n_nodos // 2])))   # extrados (succion)
    i_p = n_nodos - 1 - i_s                                     # intrados (presion)

    return {
        "delta_s": float(D[i_s, 4] * cuerda), "delta_p": float(D[i_p, 4] * cuerda),
        "theta_s": float(D[i_s, 5] * cuerda), "theta_p": float(D[i_p, 5] * cuerda),
        "Cf_s": float(D[i_s, 6]), "Cf_p": float(D[i_p, 6]),
        "H_s": float(D[i_s, 7]), "H_p": float(D[i_p, 7]),
        "Ue_s": float(D[i_s, 3]), "Ue_p": float(-D[i_p, 3]),
        "x_bl_real": float(x_c[i_s]), "n_nodos": n_nodos,
    }


def calcular_capa_limite_xfoil(perfil, cond: CondicionesOperacion,
                               alpha: float | None = None) -> dict:
    """Una corrida de XFOIL a angulo de ataque fijo: coeficientes + capa limite en el BF.

    Es la llamada que se hace **por estacion radial**, con el angulo de ataque local que
    devuelve OpenFAST y con el Re y el Mach de esa estacion (`cond` ya particularizada).
    Devuelve Cl, Cd, Cm, Cl/Cd y delta*, theta, Cf, Ue, H de extrados e intrados en
    x/c = `cond.x_bl`, que son las entradas del modelo de ruido de la seccion 5.
    """
    if alpha is not None:
        cond = replace(cond, AoA=float(alpha))
    x, y = (perfil["x"], perfil["y"]) if isinstance(perfil, dict) else perfil

    with tempfile.TemporaryDirectory(prefix="xfoil_") as tmp:
        carpeta = Path(tmp)
        escribir_dat(carpeta / "perfil.dat", x, y)
        comandos = _preambulo_xfoil("perfil.dat", cond, iteraciones=1000)
        comandos += (
            "pacc\nPOLAR_PUNTO\n\n"
            f"alfa {cond.AoA:g}\n"
            "cpwr CP_PUNTO\n"
            "DUMP DUMP_PUNTO\n"
            "pacc\n"
            "\nquit\n"
        )
        salida = _ejecutar_xfoil(comandos, carpeta, tiempo_max=TIEMPO_MAX_PUNTO)
        P = _leer_polar(carpeta / "POLAR_PUNTO")
        if len(P) == 0:
            return {"convergido": False, "alpha": float(cond.AoA)}
        punto = {
            "convergido": True,
            "alpha": float(P[-1, 0]), "Cl": float(P[-1, 1]), "Cd": float(P[-1, 2]),
            "Cdp": float(P[-1, 3]), "Cm": float(P[-1, 4]),
            "Re": float(cond.Re), "Mach": float(cond.Mach),
        }
        punto["clcd"] = punto["Cl"] / punto["Cd"] if punto["Cd"] > 0 else np.nan
        punto.update(_leer_dump(carpeta / "DUMP_PUNTO", salida, cond.x_bl, cond.cuerda))
        return punto


def _barrido(fichero_dat: str, carpeta: Path, cond: CondicionesOperacion,
             a0: float, a1: float, da: float, etiqueta: str) -> np.ndarray:
    """Una marcha ASEQ acumulada con PACC. Devuelve los puntos convergidos."""
    polar = f"POLAR_{etiqueta}"
    (carpeta / polar).unlink(missing_ok=True)
    comandos = _preambulo_xfoil(fichero_dat, cond, iteraciones=200)
    comandos += f"pacc\n{polar}\n\naseq {a0:g} {a1:g} {da:g}\npacc\n\nquit\n"
    _ejecutar_xfoil(comandos, carpeta, tiempo_max=TIEMPO_MAX_XFOIL,
                    fichero_vigilado=polar)
    return _leer_polar(carpeta / polar)


def extender_polar_viterna(alpha, cl, cd, cd_max: float = 1.3, cl_adj: float = 0.7,
                           paso: float = 1.0):
    """Extiende una polar medida a [-180, 180] deg por el metodo de Viterna.

    Dentro del rango convergido de XFOIL se interpola (PCHIP, monotona); fuera se aplican
    los coeficientes de Viterna anclados al mayor angulo medido, con el factor `cl_adj`
    para las zonas de perdida profunda y de flujo invertido (convenio de AirfoilPrep).
    """
    alpha = np.asarray(alpha, float)
    cl = np.asarray(cl, float)
    cd = np.asarray(cd, float)

    aH, aL = alpha[-1], alpha[0]                 # limites del rango medido [deg]
    ah = np.deg2rad(aH)
    clh, cdh = cl[-1], cd[-1]                    # anclaje en el extremo positivo

    A2 = (clh - cd_max * np.sin(ah) * np.cos(ah)) * np.sin(ah) / np.cos(ah) ** 2
    B2 = (cdh - cd_max * np.sin(ah) ** 2) / np.cos(ah)

    def vcl(r):
        return cd_max / 2 * np.sin(2 * r) + A2 * np.cos(r) ** 2 / max(np.sin(r), 1e-9)

    def vcd(r):
        return cd_max * np.sin(r) ** 2 + B2 * np.cos(r)

    int_cl = PchipInterpolator(alpha, cl)
    int_cd = PchipInterpolator(alpha, cd)
    piso_cd = max(cd.min(), 1e-3)

    a360 = np.arange(-180.0, 180.0 + paso / 2, paso)
    cl360 = np.zeros_like(a360)
    cd360 = np.zeros_like(a360)

    for k, ang in enumerate(a360):
        r = np.deg2rad(ang)
        if aL <= ang <= aH:                          # rango medido por XFOIL
            cl360[k], cd360[k] = int_cl(ang), int_cd(ang)
            continue
        if ang > aH and ang <= 90:                   # perdida profunda, lado +
            cl360[k], cd360[k] = vcl(r), vcd(r)
        elif 90 < ang <= 180 - aH:                   # flujo invertido, lado +
            cl360[k] = -cl_adj * vcl(np.deg2rad(180 - ang))
            cd360[k] = vcd(np.deg2rad(180 - ang))
        elif ang > 180 - aH:                         # cierre lineal hasta 0 en 180 deg
            cl360[k] = cl_adj * clh * (ang - 180) / aH
            cd360[k] = vcd(np.deg2rad(180 - ang))
        elif ang < aL and ang >= -90:                # perdida profunda, lado -
            cl360[k] = -cl_adj * vcl(np.deg2rad(-ang))
            cd360[k] = vcd(np.deg2rad(-ang))
        elif -90 > ang >= -(180 - aH):               # flujo invertido, lado -
            cl360[k] = cl_adj * vcl(np.deg2rad(180 + ang))
            cd360[k] = vcd(np.deg2rad(180 + ang))
        else:                                        # cierre lineal hasta 0 en -180 deg
            cl360[k] = cl_adj * clh * (ang + 180) / aH
            cd360[k] = vcd(np.deg2rad(180 + ang))
        cd360[k] = max(cd360[k], piso_cd)

    return a360, cl360, cd360


def calcular_aerodinamica_xfoil(perfil, cond: CondicionesOperacion = None,
                                barrido: bool = True, extender_360: bool = True,
                                rango=(-20.0, 25.0), paso_barrido: float = 0.5,
                                verbose: bool = True) -> dict:
    """Punto de referencia, polar convergida y polar +/-180 deg de una seccion.

    Parametros
      perfil        dict de `cargar_perfil`, o par (x, y) de coordenadas Selig
      cond          condiciones de esa seccion (Re y Mach salen de `cond.cuerda` y `cond.U`)
      barrido       si True, ejecuta ademas el barrido ASEQ de la polar
      extender_360  si True, extrapola la polar a +/-180 deg por Viterna
      rango         (alpha_min, alpha_max) en deg del barrido de XFOIL

    Devuelve un dict con las claves: punto, polar, polar360, cond, convergido.
    """
    cond = cond or COND
    if isinstance(perfil, dict):
        x, y = perfil["x"], perfil["y"]
        nombre = perfil["nombre"]
    else:
        x, y = perfil
        nombre = "perfil"

    resultado = {"nombre": nombre, "cond": cond,
                 "punto": calcular_capa_limite_xfoil((x, y), cond)}

    if barrido:
        with tempfile.TemporaryDirectory(prefix="xfoil_") as tmp:
            carpeta = Path(tmp)
            escribir_dat(carpeta / "perfil.dat", x, y)
            subida = _barrido("perfil.dat", carpeta, cond, 0.0, rango[1],
                              paso_barrido, "UP")
            bajada = _barrido("perfil.dat", carpeta, cond, -paso_barrido, rango[0],
                              -paso_barrido, "DN")
        P = (np.vstack([p for p in (bajada, subida) if len(p)])
             if (len(subida) or len(bajada)) else np.zeros((0, 5)))
        if len(P):
            P = np.unique(P, axis=0)                 # ordena por alpha y quita repetidos
        resultado["polar"] = P
        if extender_360 and len(P) >= 5:
            a3, cl3, cd3 = extender_polar_viterna(P[:, 0], P[:, 1], P[:, 2])
            resultado["polar360"] = np.column_stack([a3, cl3, cd3])

    resultado["convergido"] = bool(resultado["punto"].get("convergido", False))

    if verbose:
        pt = resultado["punto"]
        if pt.get("convergido"):
            print(f"[XFOIL] {nombre}: Re = {cond.Re:.2e}, M = {cond.Mach:.3f} | "
                  f"a = {pt['alpha']:.2f} deg -> Cl = {pt['Cl']:.4f}, "
                  f"Cd = {pt['Cd']:.5f}, Cl/Cd = {pt['clcd']:.2f}")
        else:
            print(f"[XFOIL] {nombre}: la corrida de referencia NO convergio.")
        if len(resultado.get("polar", [])):
            P = resultado["polar"]
            print(f"[XFOIL] {nombre}: polar convergida con {len(P)} puntos, "
                  f"alpha in [{P[:, 0].min():.1f}, {P[:, 0].max():.1f}] deg, "
                  f"Clmax = {P[:, 1].max():.3f}")
    return resultado


def condiciones_de_seccion(seccion: dict, estaciones: list,
                           cond: CondicionesOperacion = None) -> CondicionesOperacion:
    """Condiciones de XFOIL de una seccion de control: cuerda y velocidad de su estacion.

    Mientras no haya solucion de OpenFAST, la velocidad relativa se estima con la
    cinematica del rotor, `Vrel = sqrt(U_inf^2 + (Omega r)^2)`, que en una pala de
    aerogenerador domina el Reynolds local.
    """
    cond = cond or COND
    k = int(np.argmin([abs(e["r_R"] - seccion["r_R"]) for e in estaciones]))
    est = estaciones[k]
    vrel = float(np.hypot(ROTOR.U_inf, ROTOR.Omega_rad * est["r"]))
    return replace(cond, cuerda=est["cuerda"], envergadura=est["dr"], U=vrel)


def calcular_polares_pala(secciones: list, estaciones: list,
                          cond: CondicionesOperacion = None, **kwargs) -> dict:
    """Polar +/-180 de cada seccion de control, cada una con su Re y su Mach locales."""
    polares = {}
    for s in secciones:
        c = condiciones_de_seccion(s, estaciones, cond)
        polares[s["perfil"]["nombre"]] = calcular_aerodinamica_xfoil(
            s["perfil"], c, **kwargs)
    return polares


def guardar_polar(ruta: Path, datos, titulo: str, cond: CondicionesOperacion, metodo: str):
    """Escribe una polar en el formato de tres columnas alpha[deg] Cl Cd."""
    datos = np.asarray(datos)
    with Path(ruta).open("w") as f:
        f.write(f"# Polar de {titulo}\n")
        f.write(f"# Re = {cond.Re:.4g} | Mach = {cond.Mach:.3f} | "
                f"transicion forzada x/c = {cond.xtr_s:.2f} (ambas caras)\n")
        f.write(f"# N_crit = {cond.Ncrit:.3f} (Tu = {cond.Tu}) | metodo: {metodo}\n")
        f.write(f"# {'alpha[deg]':>9} {'Cl':>9} {'Cd':>9}\n")
        for fila in datos:
            f.write(f"{fila[0]:10.2f} {fila[1]:9.4f} {fila[2]:9.5f}\n")


def dibujar_polares(polares: dict, titulo: str | None = None):
    """Cl y Cd de todas las secciones: rango convergido y polar completa +/-180."""
    fig, ejes = plt.subplots(2, 2, figsize=(11.0, 6.6))
    colores = plt.cm.viridis(np.linspace(0, 0.8, len(polares)))

    for (nombre, res), color in zip(polares.items(), colores):
        P = res.get("polar", np.zeros((0, 5)))
        P3 = res.get("polar360")
        if not len(P):
            continue
        ejes[0, 0].plot(P[:, 0], P[:, 1], "-", color=color, lw=1.2, label=nombre)
        ejes[0, 1].semilogy(P[:, 0], P[:, 2], "-", color=color, lw=1.2)
        if P3 is not None:
            ejes[1, 0].plot(P3[:, 0], P3[:, 1], "-", color=color, lw=1.2, label=nombre)
            ejes[1, 1].plot(P3[:, 0], P3[:, 2], "-", color=color, lw=1.2)
        ejes[1, 0].plot(P[:, 0], P[:, 1], ".", color=color, ms=3)
        ejes[1, 1].plot(P[:, 0], P[:, 2], ".", color=color, ms=3)

    ejes[0, 0].set_xlabel(r"$\alpha$ [deg]"); ejes[0, 0].set_ylabel(r"$c_l$")
    ejes[0, 0].legend(fontsize=8); ejes[0, 0].set_title("(a) XFOIL, rango convergido")
    ejes[0, 1].set_xlabel(r"$\alpha$ [deg]"); ejes[0, 1].set_ylabel(r"$c_d$")
    ejes[0, 1].set_title("(b) XFOIL, rango convergido")
    for ax, lab, tit in ((ejes[1, 0], r"$c_l$", "(c)"), (ejes[1, 1], r"$c_d$", "(d)")):
        ax.set_xlim(-180, 180); ax.set_xticks(range(-180, 181, 60))
        ax.set_xlabel(r"$\alpha$ [deg]"); ax.set_ylabel(lab)
        ax.set_title(tit + r" polar $\pm 180^\circ$ (XFOIL + Viterna)")

    fig.suptitle(titulo or "Polares de las secciones de control")
    fig.tight_layout()
    return fig


# --------------------------------------------------------------------------------------
POLARES = calcular_polares_pala(SECCIONES_CONTROL, ESTACIONES, COND)

print("\nCondiciones aerodinamicas de referencia por seccion")
print("-" * 76)
for s in SECCIONES_CONTROL:
    res = POLARES[s["perfil"]["nombre"]]
    pt, c = res["punto"], res["cond"]
    if not pt.get("convergido"):
        print(f"{s['etiqueta']}: sin convergencia")
        continue
    print(f"{s['etiqueta']} - {s['perfil']['nombre']}  "
          f"(cuerda {c.cuerda:.2f} m, Vrel {c.U:.1f} m/s)")
    print(f"   alpha = {pt['alpha']:.2f} deg | Cl = {pt['Cl']:.4f} | "
          f"Cd = {pt['Cd']:.5f} | Cm = {pt['Cm']:.4f} | Cl/Cd = {pt['clcd']:.2f}")
    if "delta_s" in pt:
        print(f"   capa limite en x/c = {pt['x_bl_real']:.3f}: "
              f"extrados delta* = {pt['delta_s']:.3e} m, H = {pt['H_s']:.3f}, "
              f"Cf = {pt['Cf_s']:.2e} | intrados delta* = {pt['delta_p']:.3e} m, "
              f"H = {pt['H_p']:.3f}")
    guardar_polar(SALIDA / f"polar_xfoil_{s['perfil']['nombre']}.txt",
                  res["polar"][:, :3], s["perfil"]["nombre"], c,
                  "XFOIL (solo puntos convergidos)")
    if "polar360" in res:
        guardar_polar(SALIDA / f"polar360_{s['perfil']['nombre']}.txt",
                      res["polar360"], s["perfil"]["nombre"], c,
                      "XFOIL + Viterna/AirfoilPrep (CDmax = 1.3, cl_adj = 0.7)")

print(f"\nPolares +/-180 deg guardadas en {SALIDA} (insumo de AeroDyn, seccion 4)")

fig = dibujar_polares(POLARES)
fig.savefig(SALIDA / "fig_polares_secciones.png", bbox_inches="tight")
plt.show()

## 4. Analisis aeroelastico de la pala en OpenFAST: Cp y condiciones locales

**Celda plantilla, pero con el contrato de datos cerrado.** Es la pieza que convierte un
juego de perfiles en el comportamiento de la pala completa. Recibe las polares `+/-180 deg`
de la seccion 3 (que es exactamente lo que AeroDyn necesita en sus ficheros `AF_*.dat`) y la
geometria de la pala, y devuelve dos cosas que el resto de la cadena consume:

1. **Rendimiento**: `Cp`, potencia y empuje. `Cp` es la restriccion del problema: la
   reduccion de ruido no debe pagarse con perdida de produccion.
2. **Condiciones locales por estacion radial**: `alpha`, `Vrel`, `Re`, `Mach`. Con ese
   angulo de ataque local se vuelve a llamar a XFOIL en cada tira (seccion 3,
   `calcular_capa_limite_xfoil`) para obtener la capa limite con la que AMIET calcula el
   ruido. Sin este paso el ruido se estaria evaluando en un angulo de ataque inventado y no
   en el que la pala realmente ve.

Mientras la funcion no este implementada, `respuesta_cinematica_provisional` cubre su
hueco con la cinematica del rotor (`Vrel = sqrt(U_inf^2 + (Omega r)^2)`, `alpha = phi -
torsion - paso`, sin induccion): sirve para que la cadena corra de extremo a extremo y para
ver la estructura de los resultados, **no como prediccion aerodinamica**. Esta marcada como
provisional en todas las salidas.

In [ ]:
"""Acoplamiento con OpenFAST / AeroDyn: Cp y condiciones locales por estacion."""


def calcular_respuesta_openfast(polares: dict, secciones: list, estaciones: list,
                                rotor: Rotor = None, pala: Pala = None,
                                cond: CondicionesOperacion = None,
                                caso_base: Path | None = None,
                                carpeta_trabajo: Path | None = None,
                                verbose: bool = True) -> dict:
    """Ejecuta un caso de OpenFAST con las secciones dadas y devuelve rendimiento y cargas.

    Entradas
      polares      dict {nombre_seccion: resultado de `calcular_aerodinamica_xfoil`};
                   de cada uno se usa `polar360` = [alpha, Cl, Cd] para AeroDyn
      secciones    secciones de control con su posicion radial `r_R`
      estaciones   discretizacion radial de `construir_pala` (r, dr, cuerda, torsion)
      rotor, pala  punto de operacion y geometria (ROTOR, PALA)
      caso_base    carpeta con el modelo OpenFAST de referencia (.fst, AeroDyn, ElastoDyn)

    Salida: dict con este contrato, del que dependen las secciones 5 y 7
      Cp           coeficiente de potencia [-]            <- restriccion del optimizador
      potencia     potencia aerodinamica [W]
      empuje       empuje del rotor [N]
      momento_raiz momento flector en raiz [N m]
      provisional  True si no procede de una corrida real de OpenFAST
      estaciones   lista, una entrada por tira, con las claves:
                     k, r_R, r, dr, cuerda, torsion  (geometria, de `estaciones`)
                     alpha  angulo de ataque local [deg]   <- entrada de XFOIL por tira
                     Vrel   velocidad relativa local [m/s] <- Re y Mach locales
                     Re, Mach
                     Cl, Cd (de la polar de AeroDyn, para contraste)

    Implementacion pendiente:
      1. copiar `caso_base` a `carpeta_trabajo`;
      2. escribir un fichero de perfil de AeroDyn por seccion con su `polar360`
         (tabla alpha/Cl/Cd/Cm + coordenadas) y referenciarlos desde `AeroDyn15.dat`,
         con la tabla de pala (r, cuerda, torsion, id de perfil) en el fichero de pala;
      3. lanzar `openfast.exe <caso>.fst` con `subprocess` y control de tiempo;
      4. leer el `.out`/`.outb` y volcar los canales por estacion (`AB1N###Alpha`,
         `AB1N###Vrel`, `AB1N###Re`, `RtAeroCp`, `RtAeroPwr`, `RtAeroFxh`) en el
         contrato de arriba.
    """
    raise NotImplementedError(
        "calcular_respuesta_openfast: pendiente de implementar (seccion 4)."
    )


def respuesta_cinematica_provisional(estaciones: list, rotor: Rotor = None,
                                     pala: Pala = None,
                                     cond: CondicionesOperacion = None) -> dict:
    """Sustituto provisional de OpenFAST: cinematica del rotor, sin induccion.

    Calcula `Vrel` y el angulo de ataque geometrico de cada tira para que la cadena de
    evaluacion sea ejecutable antes de tener el modelo de OpenFAST. NO es una prediccion
    aerodinamica: no hay factores de induccion, ni perdidas de punta, ni aeroelasticidad,
    y `Cp` no se calcula. Todo lo que devuelve va marcado con `provisional = True`.
    """
    rotor = rotor or ROTOR
    cond = cond or COND
    salida = []
    for e in estaciones:
        u_tan = rotor.Omega_rad * e["r"]
        vrel = float(np.hypot(rotor.U_inf, u_tan))
        phi = float(np.degrees(np.arctan2(rotor.U_inf, u_tan)))   # angulo de flujo
        salida.append({
            **{k: e[k] for k in ("k", "r_R", "r", "dr", "cuerda", "torsion")},
            "alpha": phi - e["torsion"] - rotor.paso,
            "Vrel": vrel,
            "Re": e["cuerda"] * vrel / cond.nu,
            "Mach": vrel / cond.c0,
            "Cl": np.nan, "Cd": np.nan,
        })
    return {"Cp": np.nan, "potencia": np.nan, "empuje": np.nan, "momento_raiz": np.nan,
            "provisional": True, "estaciones": salida}


def openfast_disponible() -> bool:
    """True si `calcular_respuesta_openfast` ya esta implementada."""
    try:
        calcular_respuesta_openfast({}, [], [], verbose=False)
        return True
    except NotImplementedError:
        return False
    except Exception:
        return True            # implementada: el fallo es de otro tipo (entradas vacias)


OPENFAST_LISTO = openfast_disponible()


def obtener_respuesta_pala(polares: dict, secciones: list, estaciones: list,
                           verbose: bool = True) -> dict:
    """Usa OpenFAST si ya esta implementado; si no, la cinematica provisional."""
    try:
        return calcular_respuesta_openfast(polares, secciones, estaciones,
                                           verbose=verbose)
    except NotImplementedError:
        if verbose:
            print("AVISO: `calcular_respuesta_openfast` aun no esta implementada.")
            print("       Se usan condiciones locales CINEMATICAS (sin induccion) y")
            print("       Cp no disponible. Marcado como provisional en toda la cadena.")
        return respuesta_cinematica_provisional(estaciones)


# --------------------------------------------------------------------------------------
RESPUESTA = obtener_respuesta_pala(POLARES, SECCIONES_CONTROL, ESTACIONES)

print(f"\nCondiciones locales por estacion"
      f"{'  (PROVISIONALES)' if RESPUESTA.get('provisional') else ''}")
print("-" * 76)
print(f"{'k':>3}{'r/R':>7}{'cuerda':>9}{'torsion':>9}{'alpha':>9}{'Vrel':>9}"
      f"{'Re':>11}{'Mach':>7}")
for e in RESPUESTA["estaciones"]:
    print(f"{e['k']:>3d}{e['r_R']:>7.3f}{e['cuerda']:>9.3f}{e['torsion']:>9.2f}"
          f"{e['alpha']:>9.2f}{e['Vrel']:>9.1f}{e['Re']:>11.2e}{e['Mach']:>7.3f}")
cp = RESPUESTA.get("Cp")
print(f"\nCp del rotor: "
      f"{'no disponible (falta OpenFAST)' if not np.isfinite(cp) else f'{cp:.4f}'}")
if RESPUESTA.get("provisional"):
    print("\nLectura de esta tabla: los angulos de ataque de las estaciones interiores")
    print("salen muy altos porque la cinematica provisional NO incluye induccion; una")
    print("corrida real de OpenFAST los baja al entorno adherido. Sirven para comprobar")
    print("la estructura de la cadena, no para concluir nada sobre la pala.")

## 5. Ruido de borde de fuga (Amiet) por tiras e integracion sobre la pala

**Celda mixta**: la integracion sobre la pala esta escrita; lo que falta es el espectro de
una tira.

- `calcular_ruido_amiet` (**plantilla**) — nucleo fisico: a partir de los parametros de capa
  limite que XFOIL entrega en `x/c = 0.99` (`delta*`, `theta`, `Cf`, `Ue`, `H`) modela el
  espectro de presion en la superficie (TNO, Goody o Kamruzzaman), lo propaga al campo
  lejano con la teoria de Amiet y devuelve el `SPL` de **esa tira** en bandas de tercio de
  octava. Referencia de implementacion: el codigo MATLAB del proyecto
  (`work/Amiet-Theory-for-TE-Noise/`: `TNO.m`, `Integrated_WPS.m`, `farfield_noise.m`,
  `NarrowToNthOctave.m`, `Main_TE_noise_prediction.m`).
- `calcular_ruido_pala` (**operativa**) — recorre las tiras: para cada una toma el perfil
  interpolado y el `alpha`, `Vrel` y `Re` locales de OpenFAST, llama a XFOIL para la capa
  limite de esa tira, pide su espectro a AMIET y **suma en energia** los espectros de todas
  las tiras y de las `n_palas` palas, igual que `Main_Strip_theory_2D.m`. Sobre el espectro
  total aplica la ponderacion A e integra hasta el **OASPL(A) de la pala**, que es la
  funcion objetivo de la seccion 7.

La suma en energia es la parte que fija el caracter del problema: una tira ruidosa no se
compensa con otra silenciosa, y las tiras exteriores pesan mucho mas porque su velocidad
relativa es mayor. Por eso el optimo de la pala no tiene por que coincidir con el optimo de
una seccion aislada.

In [ ]:
"""Ruido de borde de fuga: espectro por tira (Amiet) e integracion sobre la pala."""

P_REF = 2e-5        # Pa, presion de referencia


def bandas_tercio_octava(f_min: float = 100.0, f_max: float = 20000.0) -> np.ndarray:
    """Frecuencias centrales de las bandas de tercio de octava (base 10, IEC 61260)."""
    n = np.arange(np.round(10 * np.log10(f_min / 1000.0)),
                  np.round(10 * np.log10(f_max / 1000.0)) + 1)
    return 1000.0 * 10.0 ** (n / 10.0)


FRECUENCIAS = bandas_tercio_octava()


def ponderacion_A(f) -> np.ndarray:
    """Correccion A-weighting [dB] en funcion de la frecuencia [Hz] (IEC 61672)."""
    f = np.asarray(f, dtype=float)
    num = (12194.0 ** 2) * f ** 4
    den = ((f ** 2 + 20.6 ** 2)
           * np.sqrt((f ** 2 + 107.7 ** 2) * (f ** 2 + 737.9 ** 2))
           * (f ** 2 + 12194.0 ** 2))
    return 20.0 * np.log10(num / den) + 2.0


def calcular_ruido_amiet(estacion: dict, capa_limite: dict,
                         cond: CondicionesOperacion = None, modelo: str = "TNO",
                         frecuencias=None, verbose: bool = False) -> dict:
    """Espectro de ruido de borde de fuga de UNA tira de la pala (teoria de Amiet).

    Entradas
      estacion     una entrada de `respuesta["estaciones"]`: r, dr, cuerda, alpha, Vrel,
                   Re, Mach. `dr` es la envergadura de la tira y `r` fija la distancia y
                   la directividad al observador.
      capa_limite  salida de `calcular_capa_limite_xfoil` para esa tira: delta*, theta,
                   Cf, Ue y H de extrados e intrados en x/c = 0.99
      cond         fluido y posicion del observador (seccion 1)
      modelo       modelo de espectro de presion en pared: TNO | Goody | Kamruzzaman
      frecuencias  bandas de analisis [Hz]; por defecto, tercio de octava 100 Hz - 20 kHz

    Salida (contrato que consume `calcular_ruido_pala`)
      f            frecuencias centrales [Hz]
      SPL          nivel de presion sonora de esta tira en el observador [dB por banda]

    Implementacion pendiente (portar desde el codigo MATLAB del proyecto):
      1. perfil de velocidad media y fluctuaciones a partir de delta*, theta, Cf y Ue;
      2. espectro de presion en la superficie Phi_pp (TNO / Goody / Kamruzzaman);
      3. longitud de correlacion transversal (Corcos) y funcion de transferencia de Amiet
         con efecto de back-scattering (Roger-Moreau 2005);
      4. integral de radiacion al observador, con la envergadura `dr` de la tira y la
         distancia y directividad que corresponden a su posicion radial;
      5. paso a bandas de tercio de octava -> SPL de la tira.
    """
    raise NotImplementedError(
        "calcular_ruido_amiet: pendiente de implementar (seccion 5)."
    )


def calcular_ruido_pala(respuesta: dict, secciones: list,
                        cond: CondicionesOperacion = None, rotor: Rotor = None,
                        modelo: str = "TNO", frecuencias=None,
                        verbose: bool = False) -> dict:
    """OASPL(A) de la pala completa: suma en energia del ruido de todas las tiras.

    Por cada tira: perfil interpolado -> XFOIL en el alpha local -> AMIET -> SPL de la
    tira. Los espectros se suman en energia sobre la envergadura y sobre las `n_palas`
    palas (mismo criterio que `Main_Strip_theory_2D.m`), y sobre el espectro total se
    aplica la ponderacion A.

    Devuelve: f, SPL, SPL_A, OASPL, OASPL_A, tiras (detalle por estacion) y n_fallos.
    """
    cond = cond or COND
    rotor = rotor or ROTOR
    f = np.asarray(frecuencias if frecuencias is not None else FRECUENCIAS, float)

    p2_total = np.zeros_like(f)          # presion cuadratica acumulada
    tiras, n_fallos = [], 0

    for est in respuesta["estaciones"]:
        perfil = interpolar_perfil(secciones, est["r_R"])
        cond_local = replace(cond, cuerda=est["cuerda"], envergadura=est["dr"],
                             U=est["Vrel"], AoA=est["alpha"])
        bl = calcular_capa_limite_xfoil(perfil, cond_local)
        if not bl.get("convergido"):
            n_fallos += 1
            tiras.append({**est, "estado": "XFOIL no convergio", "OASPL": np.nan})
            continue

        ruido = calcular_ruido_amiet(est, bl, cond_local, modelo=modelo, frecuencias=f,
                                     verbose=verbose)
        spl = np.asarray(ruido["SPL"], float)
        p2_tira = 10.0 ** (spl / 10.0)
        p2_total += rotor.n_palas * p2_tira       # las n_palas radian incoherentemente
        tiras.append({**est, "estado": "ok", "SPL": spl, "capa_limite": bl,
                      "OASPL": float(10 * np.log10(p2_tira.sum()))})

    if not np.any(p2_total > 0):
        raise RuntimeError("Ninguna tira produjo espectro: no hay ruido que integrar.")

    spl_total = 10.0 * np.log10(np.maximum(p2_total, 1e-30))
    spl_A = spl_total + ponderacion_A(f)
    return {
        "f": f, "SPL": spl_total, "SPL_A": spl_A,
        "OASPL": float(10 * np.log10(np.sum(10.0 ** (spl_total / 10.0)))),
        "OASPL_A": float(10 * np.log10(np.sum(10.0 ** (spl_A / 10.0)))),
        "tiras": tiras, "n_fallos": n_fallos,
        "provisional": bool(respuesta.get("provisional", False)),
    }


def ruido_disponible() -> bool:
    """True si `calcular_ruido_amiet` ya esta implementada."""
    est = RESPUESTA["estaciones"][0]
    try:
        calcular_ruido_amiet(est, {}, COND, verbose=False)
        return True
    except NotImplementedError:
        return False
    except Exception:
        return True            # implementada: el fallo es de otro tipo (entradas vacias)


# --------------------------------------------------------------------------------------
RUIDO_LISTO = ruido_disponible()

if RUIDO_LISTO:
    RUIDO_BASE = calcular_ruido_pala(RESPUESTA, SECCIONES_CONTROL, COND)
    print(f"OASPL de la pala : {RUIDO_BASE['OASPL']:.2f} dB | "
          f"{RUIDO_BASE['OASPL_A']:.2f} dBA")
    print(f"Tiras evaluadas  : {len(RUIDO_BASE['tiras'])} "
          f"({RUIDO_BASE['n_fallos']} sin convergencia en XFOIL)")

    fig, ejes = plt.subplots(1, 2, figsize=(11.0, 4.0))
    ejes[0].semilogx(RUIDO_BASE["f"], RUIDO_BASE["SPL"], "-o", ms=3, label="SPL")
    ejes[0].semilogx(RUIDO_BASE["f"], RUIDO_BASE["SPL_A"], "-s", ms=3, label="SPL(A)")
    ejes[0].set_xlabel("f [Hz]"); ejes[0].set_ylabel("SPL [dB]")
    ejes[0].legend(fontsize=8); ejes[0].set_title("(a) Espectro de la pala")
    r_R = [t["r_R"] for t in RUIDO_BASE["tiras"]]
    ejes[1].plot(r_R, [t["OASPL"] for t in RUIDO_BASE["tiras"]], "-o", ms=3)
    ejes[1].set_xlabel("r/R"); ejes[1].set_ylabel("OASPL de la tira [dB]")
    ejes[1].set_title("(b) Reparto radial del ruido")
    fig.tight_layout()
    fig.savefig(SALIDA / "fig_ruido_pala.png", bbox_inches="tight")
    plt.show()
else:
    print("`calcular_ruido_amiet` aun no esta implementada (seccion 5).")
    print("La integracion sobre la pala (`calcular_ruido_pala`) ya esta escrita y se")
    print("activara sola en cuanto la plantilla devuelva el SPL de una tira.")
    print(f"\nMalla de frecuencias preparada: {len(FRECUENCIAS)} bandas de tercio de "
          f"octava, {FRECUENCIAS[0]:.0f} Hz - {FRECUENCIAS[-1]/1000:.1f} kHz")

## 6. Parametrizacion de las siluetas: CST (Kulfan) y PARSEC

Para optimizar hay que reducir los ~200 puntos del contorno de **cada seccion de control** a
un puñado de variables. El vector de diseno del problema es la concatenacion de las
variables de todas las secciones: con dos secciones y CST de orden 6 son `2 x 14 = 28`
variables; con PARSEC, `2 x 11 = 22`.

**CST (Kulfan)** — cada cara se escribe como `y(x) = C(x)*S(x) + x*y_te`, con la funcion de
clase `C(x) = x^N1 (1-x)^N2` (`N1 = 0.5`, `N2 = 1.0`: borde de ataque redondo y borde de
fuga afilado) y la funcion de forma `S(x)` como combinacion de polinomios de Bernstein de
orden `n`. Variables por seccion: `2*(n+1)`. Los extremos `x = 0` y `x = 1` se excluyen del
ajuste porque la funcion de clase se anula alli.

**PARSEC** — 11 parametros con significado geometrico directo (radio de borde de ataque,
posicion/valor/curvatura de las crestas de extrados e intrados, posicion y apertura del
borde de fuga, angulo de salida y angulo de cuña). Cada cara es `y(x) = sum a_n x^(n-1/2)`
con los 6 coeficientes resueltos a partir de esas restricciones. Se incluye tambien un
ajuste por minimos cuadrados sobre la misma base, que mide la capacidad de representacion
de la base sin imponer las restricciones geometricas.

> En perfiles con intrados reflexado o muy cargado el PARSEC clasico puede divergir porque
> su hipotesis de cresta unica deja de cumplirse; el error RMS que imprime la celda lo
> evidencia para cada seccion. La variable `PARAMETRIZACION` de la seccion 7 selecciona
> cual se usa en la optimizacion.

In [ ]:
"""Parametrizaciones CST (Kulfan) y PARSEC: ajuste y comparacion en cada seccion."""

# =======================================================================================
# CST (Kulfan)
# =======================================================================================
def cst_eval(x, A, yte: float, N1: float = 0.5, N2: float = 1.0) -> np.ndarray:
    """Evalua una cara CST: y = C(x)*S(x) + x*yte."""
    x = np.asarray(x, float).ravel()
    A = np.asarray(A, float).ravel()
    n = len(A) - 1
    C = x ** N1 * (1.0 - x) ** N2                        # funcion de clase
    S = np.zeros_like(x)
    for i in range(n + 1):                               # funcion de forma (Bernstein)
        S += A[i] * comb(n, i) * x ** i * (1.0 - x) ** (n - i)
    return C * S + x * yte


def _cst_ajustar_cara(x, y, n: int, N1: float, N2: float) -> np.ndarray:
    """Minimos cuadrados de los coeficientes de Bernstein de una cara."""
    x = np.asarray(x, float).ravel()
    y = np.asarray(y, float).ravel()
    C = x ** N1 * (1.0 - x) ** N2
    yte = y[-1]
    rhs = y - x * yte
    B = np.column_stack([comb(n, i) * x ** i * (1.0 - x) ** (n - i) for i in range(n + 1)])
    M = C[:, None] * B
    mask = C > 1e-12                                     # descarta x = 0 y x = 1
    return np.linalg.lstsq(M[mask], rhs[mask], rcond=None)[0]


def cst_fit(perfil: dict, orden: int = 6, N1: float = 0.5, N2: float = 1.0) -> dict:
    """Ajusta coeficientes CST al perfil. Variables de diseno: 2*(orden+1)."""
    return {
        "tipo": "CST",
        "Au": _cst_ajustar_cara(perfil["xu"], perfil["yu"], orden, N1, N2),
        "Al": _cst_ajustar_cara(perfil["xl"], perfil["yl"], orden, N1, N2),
        "yte_u": float(perfil["yu"][-1]), "yte_l": float(perfil["yl"][-1]),
        "N1": N1, "N2": N2, "orden": orden, "ndv": 2 * (orden + 1),
    }


def cst_to_coords(p: dict, xs=None):
    """Caras del perfil sobre la malla comun a partir de los coeficientes CST."""
    xs = X_MALLA if xs is None else np.asarray(xs, float)
    yu = cst_eval(xs, p["Au"], p["yte_u"], p["N1"], p["N2"])
    yl = cst_eval(xs, p["Al"], p["yte_l"], p["N1"], p["N2"])
    return xs, yu, yl


# =======================================================================================
# PARSEC
# =======================================================================================
def parsec_coeffs(a1: float, Xm: float, Zm: float, Zxx: float,
                  Zte: float, pendiente_te: float) -> np.ndarray:
    """Resuelve los 6 coeficientes de una cara PARSEC: y(x) = sum a_n x^(n-1/2).

    Restricciones: coeficiente de borde de ataque, y(Xm) = Zm, y'(Xm) = 0, y''(Xm) = Zxx,
    y(1) = Zte, y'(1) = pendiente_te.
    """
    e = np.arange(1, 7) - 0.5                            # exponentes 0.5 ... 5.5
    M = np.zeros((6, 6))
    b = np.zeros(6)
    M[0], b[0] = [1, 0, 0, 0, 0, 0], a1
    M[1], b[1] = Xm ** e, Zm
    M[2], b[2] = e * Xm ** (e - 1), 0.0
    M[3], b[3] = e * (e - 1) * Xm ** (e - 2), Zxx
    M[4], b[4] = np.ones(6), Zte
    M[5], b[5] = e, pendiente_te
    return np.linalg.solve(M, b)


def parsec_eval(x, a) -> np.ndarray:
    """Evalua una cara PARSEC: y(x) = sum_{n=1..6} a_n x^(n-1/2)."""
    x = np.asarray(x, float).ravel()
    e = np.arange(1, 7) - 0.5
    return (x[:, None] ** e) @ np.asarray(a, float).ravel()


def _curvatura_local(x, y, i: int) -> float:
    """Segunda derivada por ajuste cuadratico local alrededor del indice i."""
    lo, hi = max(0, i - 2), min(len(x), i + 3)
    return float(2.0 * np.polyfit(x[lo:hi], y[lo:hi], 2)[0])


ORDEN_FEAT_PARSEC = ["rLE", "Xup", "Zup", "Zxxup", "Xlo", "Zlo", "Zxxlo",
                     "Zte", "dZte", "aTE", "bTE"]


def parsec_fit(perfil: dict) -> dict:
    """Extrae los 11 parametros geometricos PARSEC y construye los coeficientes.

    PARSEC es una parametrizacion de rasgos geometricos, de modo que se miden directamente
    sobre las coordenadas (uso estandar) en lugar de un ajuste ciego por minimos cuadrados.
    Se devuelve tambien el ajuste LSQ sobre la misma base como referencia.
    """
    xu, yu = np.asarray(perfil["xu"]), np.asarray(perfil["yu"])
    xl, yl = np.asarray(perfil["xl"]), np.asarray(perfil["yl"])

    cerca = (xu > 0) & (xu < 0.05)                        # radio de borde de ataque
    a1u_est = float(np.linalg.lstsq(np.sqrt(xu[cerca])[:, None], yu[cerca],
                                    rcond=None)[0][0])
    rLE = a1u_est ** 2 / 2.0

    iu = int(np.argmax(yu))                               # cresta del extrados
    il = int(np.argmin(yl))                               # cresta del intrados
    feat = {
        "rLE": rLE,
        "Xup": float(xu[iu]), "Zup": float(yu[iu]), "Zxxup": _curvatura_local(xu, yu, iu),
        "Xlo": float(xl[il]), "Zlo": float(yl[il]), "Zxxlo": _curvatura_local(xl, yl, il),
        "Zte": float((yu[-1] + yl[-1]) / 2.0), "dZte": float(yu[-1] - yl[-1]),
    }
    m_u = (yu[-1] - yu[-2]) / (xu[-1] - xu[-2])
    m_l = (yl[-1] - yl[-2]) / (xl[-1] - xl[-2])
    feat["aTE"] = float(np.arctan((m_u + m_l) / 2.0))     # angulo de salida
    feat["bTE"] = float(np.arctan(m_u) - np.arctan(m_l))  # angulo de cuña

    p = {"tipo": "PARSEC", "feat": feat, "ndv": 11, "orden_feat": ORDEN_FEAT_PARSEC}
    p.update(parsec_desde_features(feat))

    e = np.arange(1, 7) - 0.5
    p["au_lsq"] = np.linalg.lstsq(xu[:, None] ** e, yu, rcond=None)[0]
    p["al_lsq"] = np.linalg.lstsq(xl[:, None] ** e, yl, rcond=None)[0]
    return p


def parsec_desde_features(feat: dict) -> dict:
    """Coeficientes de ambas caras a partir de los 11 parametros PARSEC."""
    rLE = max(feat["rLE"], 1e-8)
    a1 = np.sqrt(2.0 * rLE)
    zte_u = feat["Zte"] + feat["dZte"] / 2.0
    zte_l = feat["Zte"] - feat["dZte"] / 2.0
    m_u = np.tan(feat["aTE"] + feat["bTE"] / 2.0)
    m_l = np.tan(feat["aTE"] - feat["bTE"] / 2.0)
    return {
        "au": parsec_coeffs(a1, feat["Xup"], feat["Zup"], feat["Zxxup"], zte_u, m_u),
        "al": parsec_coeffs(-a1, feat["Xlo"], feat["Zlo"], feat["Zxxlo"], zte_l, m_l),
    }


def parsec_to_coords(p: dict, xs=None):
    """Caras del perfil sobre la malla comun a partir de los coeficientes PARSEC."""
    xs = X_MALLA if xs is None else np.asarray(xs, float)
    return xs, parsec_eval(xs, p["au"]), parsec_eval(xs, p["al"])


# =======================================================================================
# Ajuste de todas las secciones de control y comparacion
# =======================================================================================
def error_reconstruccion(perfil: dict, xs, yu_rec, yl_rec) -> float:
    """RMS entre el perfil original y su reconstruccion, sobre la misma malla."""
    yu_ref = PchipInterpolator(perfil["xu"], perfil["yu"])(xs)
    yl_ref = PchipInterpolator(perfil["xl"], perfil["yl"])(xs)
    return float(np.sqrt(np.mean(np.concatenate([yu_rec - yu_ref, yl_rec - yl_ref]) ** 2)))


ORDEN_CST = 6

fig, ejes = plt.subplots(2, len(SECCIONES_CONTROL), figsize=(5.6 * len(SECCIONES_CONTROL),
                                                             6.0),
                         squeeze=False, gridspec_kw={"height_ratios": [2, 1]})

print(f"{'seccion':<16}{'parametrizacion':<28}{'n var':>7}{'RMS y/c':>13}")
print("-" * 66)
for j, s in enumerate(SECCIONES_CONTROL):
    p = s["perfil"]
    s["cst"] = cst_fit(p, orden=ORDEN_CST)
    s["parsec"] = parsec_fit(p)

    xs, yu_c, yl_c = cst_to_coords(s["cst"])
    _, yu_p, yl_p = parsec_to_coords(s["parsec"])
    yu_lsq = parsec_eval(xs, s["parsec"]["au_lsq"])
    yl_lsq = parsec_eval(xs, s["parsec"]["al_lsq"])

    rms = {
        "CST": error_reconstruccion(p, xs, yu_c, yl_c),
        "PARSEC (geometrico)": error_reconstruccion(p, xs, yu_p, yl_p),
        "PARSEC (LSQ sobre la base)": error_reconstruccion(p, xs, yu_lsq, yl_lsq),
    }
    s["rms"] = rms
    ndv = {"CST": s["cst"]["ndv"], "PARSEC (geometrico)": s["parsec"]["ndv"],
           "PARSEC (LSQ sobre la base)": 12}
    for k, v in rms.items():
        print(f"{p['nombre'] if k == 'CST' else '':<16}{k:<28}{ndv[k]:>7}{v:>13.2e}")

    ax = ejes[0, j]
    ax.plot(p["x"], p["y"], "o", color="0.3", ms=2.5, label="original")
    ax.plot(np.concatenate([xs[::-1], xs[1:]]),
            np.concatenate([yu_c[::-1], yl_c[1:]]), "-", color="tab:blue", lw=1.3,
            label=f"CST orden {ORDEN_CST} ({rms['CST']:.1e})")
    ax.plot(np.concatenate([xs[::-1], xs[1:]]),
            np.concatenate([yu_p[::-1], yl_p[1:]]), "--", color="tab:red", lw=1.3,
            label=f"PARSEC ({rms['PARSEC (geometrico)']:.1e})")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-0.02, 1.02)
    ax.set_xlabel("x/c"); ax.set_ylabel("y/c")
    ax.legend(fontsize=7, loc="upper right")
    ax.set_title(f"{s['etiqueta']}: {p['nombre']}")

    ax = ejes[1, j]
    yu_ref = PchipInterpolator(p["xu"], p["yu"])(xs)
    yl_ref = PchipInterpolator(p["xl"], p["yl"])(xs)
    ax.plot(xs, 1e3 * (yu_c - yu_ref), "-", color="tab:blue", label="CST, extrados")
    ax.plot(xs, 1e3 * (yl_c - yl_ref), "--", color="tab:blue", label="CST, intrados")
    ax.plot(xs, 1e3 * (yu_p - yu_ref), "-", color="tab:red", label="PARSEC, extrados")
    ax.plot(xs, 1e3 * (yl_p - yl_ref), "--", color="tab:red", label="PARSEC, intrados")
    ax.set_xlabel("x/c"); ax.set_ylabel(r"error $\times 10^{3}$ (y/c)")
    ax.legend(fontsize=6, ncol=2)

fig.tight_layout()
fig.savefig(SALIDA / "fig_parametrizacion_secciones.png", bbox_inches="tight")
plt.show()

print(f"\nVariables de diseno del problema completo "
      f"({len(SECCIONES_CONTROL)} secciones):")
print(f"  CST orden {ORDEN_CST}: "
      f"{sum(s['cst']['ndv'] for s in SECCIONES_CONTROL)} variables")
print(f"  PARSEC          : "
      f"{sum(s['parsec']['ndv'] for s in SECCIONES_CONTROL)} variables")

## 7. Optimizacion con Dual Annealing sobre el OASPL de la pala

Se cierra el lazo del esquema de trabajo:

```
dv  ->  siluetas de X1 y X2  ->  XFOIL (polares +/-180)  ->  OpenFAST (Cp, alpha/Vrel por tira)
                                                                            |
        J = OASPL_A(pala) + penalizaciones   <---  AMIET por tira + suma sobre la pala
```

- **Variables de diseno**: la concatenacion de los coeficientes de las dos secciones de
  control (28 con CST de orden 6, 22 con PARSEC). Mover una variable cambia la pala entera,
  porque las estaciones intermedias interpolan entre `X1` y `X2`.
- **Dominio**: caja alrededor de las siluetas de partida, de semiancho
  `max(0.05, 0.5*|dv0|)` en CST y `25 %` relativo en PARSEC.
- **Objetivo**: `OASPL_A` de la pala completa, el que devuelve `calcular_ruido_pala`.
- **Restricciones**, por penalizacion: geometria valida de cada seccion (sin cruce de
  caras, espesor entre el 50 % y el 100 % del de partida) y **`Cp` no inferior al de la
  pala de partida**, que es la restriccion de rendimiento del esquema. Mientras OpenFAST no
  devuelva `Cp`, se usa como sustituto el `Cl/Cd` medio de las secciones.
- **Algoritmo**: `scipy.optimize.dual_annealing`, recocido simulado generalizado con
  busqueda local: global y sin gradiente, que es lo que pide un evaluador ruidoso y con
  fallos de convergencia como este. Arranca desde la pala de partida con `x0`.

**Coste.** Cada evaluacion del objetivo son varias corridas de XFOIL: las polares de las
secciones de control y, si el ruido esta activo, una corrida por tira en su angulo de ataque
local. El tope `MAX_EVALUACIONES` se impone dentro del evaluador, no solo con el `maxfun` de
SciPy: `dual_annealing` no interrumpe su busqueda local al agotarse el presupuesto y
llegaria a multiplicar por diez el numero de corridas. Ademas se memorizan los resultados y
los disenos geometricamente invalidos se rechazan sin gastar presupuesto.

> Si todavia faltan OpenFAST (seccion 4) o AMIET (seccion 5), la celda lo detecta y avisa:
> optimiza con un **objetivo sustituto** (`-Cl/Cd` medio de las secciones) para que el
> andamiaje sea ejecutable de extremo a extremo. Al implementar esas dos celdas pasa sola a
> minimizar el OASPL(A) de la pala, sin tocar nada mas.

In [ ]:
"""Optimizacion de las siluetas de las secciones de control minimizando el ruido."""

# ---------------------------------------------------------------------------------------
# Ajustes de la corrida
# ---------------------------------------------------------------------------------------
PARAMETRIZACION = "CST"       # "CST" o "PARSEC"
MODO_OBJETIVO = "auto"        # "auto" | "ruido" | "clcd"
MAX_EVALUACIONES = 60         # presupuesto de evaluaciones completas de la pala
SEMILLA = 0
PENALIZACION = 200.0          # valor asignado a los disenos invalidos o no convergidos
# Semiancho de la caja de busqueda alrededor de la silueta de partida. Estrecharla sube la
# tasa de aceptacion geometrica (que la celda imprime al final) a costa de explorar menos.
ANCHO_REL = 0.50              # fraccion del valor de cada coeficiente (CST)
ANCHO_ABS = 0.05              # semiancho minimo absoluto (CST)
ANCHO_REL_PARSEC = 0.25       # fraccion del valor de cada parametro (PARSEC)
PESO_RENDIMIENTO = 5.0        # dBA por unidad de deficit de Cp (o de Cl/Cd sustituto)
TOL_CRUCE = 1e-4              # holgura de espesor negativo admitida
TOL_ESPESOR = 1e-3            # holgura sobre el techo de espesor

# ---------------------------------------------------------------------------------------
# Variables de diseno <-> siluetas de las secciones de control
# ---------------------------------------------------------------------------------------
def variables_iniciales(parametrizacion: str = PARAMETRIZACION):
    """Vector de diseno de partida (todas las secciones concatenadas), cotas e indices."""
    dv0, semiancho, indices, i0 = [], [], [], 0
    for s in SECCIONES_CONTROL:
        if parametrizacion == "CST":
            v = np.concatenate([s["cst"]["Au"], s["cst"]["Al"]])
            w = np.maximum(ANCHO_ABS, ANCHO_REL * np.abs(v))
        elif parametrizacion == "PARSEC":
            v = np.array([s["parsec"]["feat"][k] for k in ORDEN_FEAT_PARSEC], float)
            w = np.maximum(ANCHO_REL_PARSEC * np.abs(v), 1e-3)
        else:
            raise ValueError("PARAMETRIZACION debe ser 'CST' o 'PARSEC'.")
        dv0.append(v)
        semiancho.append(w)
        indices.append((i0, i0 + len(v)))
        i0 += len(v)
    dv0 = np.concatenate(dv0)
    semiancho = np.concatenate(semiancho)
    return dv0, dv0 - semiancho, dv0 + semiancho, indices


def construir_secciones_desde_dv(dv, parametrizacion: str = PARAMETRIZACION) -> list:
    """Reconstruye las secciones de control a partir del vector de diseno completo."""
    dv = np.asarray(dv, float)
    secciones = []
    for s, (a, b) in zip(SECCIONES_CONTROL, INDICES):
        v = dv[a:b]
        if parametrizacion == "CST":
            n = len(v) // 2
            p = dict(s["cst"], Au=v[:n], Al=v[n:])
            xs, yu, yl = cst_to_coords(p)
        else:
            feat = dict(zip(ORDEN_FEAT_PARSEC, v))
            p = dict(s["parsec"], **parsec_desde_features(feat))
            xs, yu, yl = parsec_to_coords(p)
        perfil = perfil_desde_caras(s["perfil"]["nombre"], xs, yu, yl)
        secciones.append({**s, "perfil": perfil})
    return secciones


def geometria_valida(perfil: dict, t_min: float, t_max: float) -> str:
    """Devuelve "" si la silueta es admisible, o el motivo del rechazo."""
    espesor = perfil["yu"] - perfil["yl"]
    if not np.all(np.isfinite(espesor)):
        return "geometria no finita"
    if np.min(espesor) < -TOL_CRUCE:
        return "caras cruzadas"
    e = float(np.max(espesor))
    if e < t_min:
        return f"demasiado delgada (t = {100 * e:.1f}% c)"
    if e > t_max + TOL_ESPESOR:
        return f"demasiado gruesa (t = {100 * e:.1f}% c)"
    return ""


# ---------------------------------------------------------------------------------------
# Evaluador de la pala, con memoria y registro
# ---------------------------------------------------------------------------------------
_CACHE: dict = {}
HISTORIAL: list = []
N_EVAL = 0                    # evaluaciones completas de la pala consumidas


def evaluar_pala(dv, parametrizacion: str = PARAMETRIZACION) -> dict:
    """Siluetas -> XFOIL -> OpenFAST -> AMIET. Devuelve el objetivo y sus componentes."""
    global N_EVAL
    clave = tuple(np.round(np.asarray(dv, float), 10))
    if clave in _CACHE:
        return _CACHE[clave]

    secciones = construir_secciones_desde_dv(dv, parametrizacion)

    for s, t_lim in zip(secciones, LIMITES_ESPESOR):          # rechazo geometrico gratuito
        motivo = geometria_valida(s["perfil"], *t_lim)
        if motivo:
            R = {"valido": False, "estado": f"{s['etiqueta']}: {motivo}",
                 "J": PENALIZACION, "OASPL_A": np.nan, "Cp": np.nan, "clcd": np.nan}
            _CACHE[clave] = R
            return R

    # `dual_annealing` no interrumpe su busqueda local al agotarse `maxfun`, de modo que el
    # presupuesto se impone aqui: agotado, se devuelve la penalizacion sin evaluar nada.
    # El optimizador conserva el mejor diseno ya encontrado, asi que el resultado no cambia.
    if N_EVAL >= MAX_EVALUACIONES:
        return {"valido": False, "estado": "presupuesto agotado", "J": PENALIZACION,
                "OASPL_A": np.nan, "Cp": np.nan, "clcd": np.nan}
    N_EVAL += 1

    # 1) polares de las secciones (el barrido +/-180 solo hace falta si OpenFAST lo usa)
    polares = calcular_polares_pala(secciones, ESTACIONES, COND,
                                    barrido=OPENFAST_LISTO,
                                    extender_360=OPENFAST_LISTO, verbose=False)
    if not all(r["convergido"] for r in polares.values()):
        R = {"valido": False, "estado": "XFOIL no convergio en alguna seccion",
             "J": PENALIZACION, "OASPL_A": np.nan, "Cp": np.nan, "clcd": np.nan}
        _CACHE[clave] = R
        return R
    clcd = float(np.mean([r["punto"]["clcd"] for r in polares.values()]))

    # 2) pala completa: Cp y condiciones locales por tira
    respuesta = obtener_respuesta_pala(polares, secciones, ESTACIONES, verbose=False)
    cp = respuesta.get("Cp", np.nan)

    # 3) ruido de la pala
    if OBJETIVO == "ruido":
        ruido = calcular_ruido_pala(respuesta, secciones, COND)
        merito = float(ruido["OASPL_A"])
    else:
        ruido = None
        merito = -clcd                        # objetivo sustituto: maximizar Cl/Cd medio

    # 4) restriccion de rendimiento: Cp si lo hay, Cl/Cd medio como sustituto
    if np.isfinite(cp) and np.isfinite(CP_BASE):
        deficit = max(0.0, CP_BASE - cp) / max(CP_BASE, 1e-6) * 100.0   # % de Cp perdido
    else:
        deficit = max(0.0, CLCD_BASE - clcd)

    R = {"valido": True, "estado": "ok", "J": merito + PESO_RENDIMIENTO * deficit,
         "merito": merito, "OASPL_A": (ruido["OASPL_A"] if ruido else np.nan),
         "Cp": cp, "clcd": clcd, "deficit": deficit,
         "t_max": [float(np.max(s["perfil"]["yu"] - s["perfil"]["yl"]))
                   for s in secciones],
         "polares": polares, "respuesta": respuesta, "ruido": ruido}
    _CACHE[clave] = R
    return R


def funcion_objetivo(dv) -> float:
    """Funcion escalar que minimiza el optimizador; registra el historial."""
    R = evaluar_pala(dv)
    mejor = min([h["J"] for h in HISTORIAL], default=np.inf)
    HISTORIAL.append({"n": len(HISTORIAL) + 1, "J": R["J"], "mejor": min(mejor, R["J"]),
                      "OASPL_A": R["OASPL_A"], "Cp": R["Cp"], "clcd": R["clcd"],
                      "estado": R["estado"], "dv": np.asarray(dv, float).copy()})
    return R["J"]


# ---------------------------------------------------------------------------------------
# Preparacion: modo de objetivo, pala de referencia y dominio de busqueda
# ---------------------------------------------------------------------------------------
if MODO_OBJETIVO == "auto":
    OBJETIVO = "ruido" if RUIDO_LISTO else "clcd"
else:
    OBJETIVO = MODO_OBJETIVO

if OBJETIVO == "clcd":
    print("AVISO: la cadena de ruido aun no esta completa.")
    if not OPENFAST_LISTO:
        print("       - falta `calcular_respuesta_openfast` (seccion 4)")
    if not RUIDO_LISTO:
        print("       - falta `calcular_ruido_amiet` (seccion 5)")
    print("       Se optimiza con el objetivo SUSTITUTO -Cl/Cd medio de las secciones,")
    print("       NO con el ruido de la pala. Al completar esas celdas, esta pasa sola")
    print("       a minimizar el OASPL(A) de la pala.\n")

dv0, lb, ub, INDICES = variables_iniciales(PARAMETRIZACION)
secciones_base = construir_secciones_desde_dv(dv0, PARAMETRIZACION)
LIMITES_ESPESOR = [(0.5 * float(np.max(s["perfil"]["yu"] - s["perfil"]["yl"])),
                    float(np.max(s["perfil"]["yu"] - s["perfil"]["yl"])))
                   for s in secciones_base]
CP_BASE = -np.inf                            # provisional, para evaluar la referencia
CLCD_BASE = -np.inf

R0 = evaluar_pala(dv0)
if not R0["valido"]:
    raise RuntimeError(f"La pala de partida no se pudo evaluar: {R0['estado']}")
# Suelos de rendimiento = los de la pala de partida. Su propio deficit es nulo, de modo que
# fijarlos despues de evaluarla no altera su objetivo y evita repetir la corrida.
CP_BASE = R0["Cp"]
CLCD_BASE = R0["clcd"]
HISTORIAL.clear()
N_EVAL = 0

print(f"Parametrizacion   : {PARAMETRIZACION}, {len(SECCIONES_CONTROL)} secciones "
      f"-> {len(dv0)} variables de diseno")
print(f"Objetivo          : "
      f"{'OASPL(A) de la pala [dBA]' if OBJETIVO == 'ruido' else '-Cl/Cd medio (sustituto)'}")
print(f"Referencia        : J0 = {R0['J']:.4f} | "
      f"OASPL_A = {R0['OASPL_A'] if np.isfinite(R0['OASPL_A']) else float('nan'):.2f} dBA | "
      f"Cp = {R0['Cp'] if np.isfinite(R0['Cp']) else float('nan'):.4f} | "
      f"Cl/Cd medio = {R0['clcd']:.2f}")
for s, t_lim, t in zip(SECCIONES_CONTROL, LIMITES_ESPESOR, R0["t_max"]):
    print(f"  {s['etiqueta']:<18} espesor {100 * t:.2f}% c, "
          f"admisible [{100 * t_lim[0]:.2f}, {100 * t_lim[1]:.2f}] % c")
print(f"Presupuesto       : {MAX_EVALUACIONES} evaluaciones completas de la pala\n")

# ---------------------------------------------------------------------------------------
# Corrida del optimizador
# ---------------------------------------------------------------------------------------
resultado = dual_annealing(
    funcion_objetivo,
    bounds=list(zip(lb, ub)),
    x0=dv0,
    # El presupuesto real lo lleva `evaluar_pala` (los rechazos geometricos no cuestan);
    # a SciPy se le da un tope holgado para que no agote su cuenta con disenos rechazados.
    maxfun=max(2000, 50 * MAX_EVALUACIONES),
    maxiter=1000,
    seed=SEMILLA,
    initial_temp=5230.0,
    visit=2.62,
    accept=-5.0,
    no_local_search=False,
)

dv_opt = resultado.x
R_opt = evaluar_pala(dv_opt)
secciones_opt = construir_secciones_desde_dv(dv_opt, PARAMETRIZACION)

mensaje = resultado.message
if not isinstance(mensaje, str):
    mensaje = "; ".join(mensaje)

# Las llamadas posteriores al agotamiento del presupuesto no aportan informacion: se
# separan de los rechazos reales (geometria invalida o XFOIL sin convergencia).
HIST = [h for h in HISTORIAL if h["estado"] != "presupuesto agotado"]
n_rechazados = sum(1 for h in HIST if h["J"] >= PENALIZACION)
n_sin_presupuesto = len(HISTORIAL) - len(HIST)

print("\nResultado de la optimizacion")
print("-" * 70)
print(f"Evaluaciones      : {N_EVAL} evaluaciones completas de la pala, "
      f"{n_rechazados} disenos rechazados "
      f"({100 * N_EVAL / max(len(HIST), 1):.0f} % de aceptacion)")
if n_sin_presupuesto:
    print(f"                    (+{n_sin_presupuesto} llamadas tras agotar el "
          f"presupuesto, sin coste)")
print(f"Criterio de parada: {mensaje}")
print(f"Objetivo          : {R0['J']:.4f}  ->  {R_opt['J']:.4f}  "
      f"(mejora {R0['J'] - R_opt['J']:+.4f})")
if OBJETIVO == "ruido":
    print(f"OASPL(A) de pala  : {R0['OASPL_A']:.2f}  ->  {R_opt['OASPL_A']:.2f} dBA  "
          f"({R_opt['OASPL_A'] - R0['OASPL_A']:+.2f} dBA)")
if np.isfinite(R_opt["Cp"]):
    print(f"Cp                : {R0['Cp']:.4f}  ->  {R_opt['Cp']:.4f}")
print(f"Cl/Cd medio       : {R0['clcd']:.2f}  ->  {R_opt['clcd']:.2f}")
for s, t0, t1 in zip(SECCIONES_CONTROL, R0["t_max"], R_opt["t_max"]):
    print(f"  {s['etiqueta']:<18} espesor {100 * t0:.2f}%  ->  {100 * t1:.2f}% c")

for s in secciones_opt:
    ruta = SALIDA / f"perfil_optimizado_{s['perfil']['nombre']}.dat"
    escribir_dat(ruta, s["perfil"]["x"], s["perfil"]["y"])
    print(f"Silueta optima guardada: {ruta.name}")

if RESPUESTA.get("provisional"):
    print("\nRECORDATORIO: las condiciones por tira son provisionales (sin OpenFAST),")
    print("              de modo que estos numeros validan la cadena, no la pala.")

# ---------------------------------------------------------------------------------------
# Figuras de sintesis
# ---------------------------------------------------------------------------------------
n_sec = len(SECCIONES_CONTROL)
fig = plt.figure(figsize=(11.0, 8.6))
for j, (s0, s1) in enumerate(zip(secciones_base, secciones_opt)):
    ax = fig.add_subplot(3, n_sec, j + 1)
    ax.plot(s0["perfil"]["x"], s0["perfil"]["y"], "-", color="0.25", lw=1.4,
            label="partida")
    ax.plot(s1["perfil"]["x"], s1["perfil"]["y"], "-", color="tab:red", lw=1.4,
            label="optimizada")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-0.02, 1.02)
    ax.set_xlabel("x/c"); ax.set_ylabel("y/c")
    ax.legend(fontsize=7, loc="upper right")
    ax.set_title(f"({chr(97 + j)}) {s0['etiqueta']}: {s0['perfil']['nombre']}")

ax = fig.add_subplot(3, 1, 2)
# El eje son las evaluaciones que realmente costaron XFOIL; los disenos rechazados no
# consumen presupuesto y solo se resumen en la leyenda.
J_val = [h["J"] for h in HIST if h["J"] < PENALIZACION]
n_val = np.arange(1, len(J_val) + 1)
mejor = np.minimum.accumulate(J_val) if J_val else np.array([])
ax.plot(n_val, J_val, ".", color="0.6", ms=6,
        label=f"disenos admisibles ({N_EVAL} evaluados, el resto servido de cache; "
              f"{n_rechazados} rechazados sin coste)")
ax.plot(n_val, mejor, "-", color="tab:red", lw=1.4, label="mejor hasta el momento")
if J_val:
    lo, hi = min(J_val), max(J_val)
    margen = max(0.05 * (hi - lo), 0.5)
    ax.set_ylim(lo - margen, hi + margen)
ax.set_xlabel("evaluacion completa de la pala")
ax.set_ylabel("OASPL(A) pala [dBA]" if OBJETIVO == "ruido" else r"$-c_l/c_d$ medio")
ax.legend(fontsize=8)
ax.set_title(f"({chr(97 + n_sec)}) Historial de convergencia del dual annealing")

ax = fig.add_subplot(3, 1, 3)
if OBJETIVO == "ruido" and R_opt.get("ruido") and R0.get("ruido"):
    ax.semilogx(R0["ruido"]["f"], R0["ruido"]["SPL_A"], "-", color="0.25",
                label="pala de partida")
    ax.semilogx(R_opt["ruido"]["f"], R_opt["ruido"]["SPL_A"], "-", color="tab:red",
                label="pala optimizada")
    ax.set_xlabel("f [Hz]"); ax.set_ylabel("SPL(A) [dB]")
    ax.set_title(f"({chr(98 + n_sec)}) Espectro ponderado A de la pala")
else:
    # Sin espectro todavia: se compara lo que si cambia y explica el resultado, la
    # distribucion de espesor de cada seccion.
    for s0, s1, estilo in zip(secciones_base, secciones_opt, ("-", "--")):
        ax.plot(X_MALLA, 100 * (s0["perfil"]["yu"] - s0["perfil"]["yl"]), estilo,
                color="0.25", lw=1.3, label=f"{s0['etiqueta']} partida")
        ax.plot(X_MALLA, 100 * (s1["perfil"]["yu"] - s1["perfil"]["yl"]), estilo,
                color="tab:red", lw=1.3, label=f"{s1['etiqueta']} optimizada")
    ax.set_xlabel("x/c"); ax.set_ylabel("espesor [% c]")
    ax.legend(fontsize=7, ncol=2)
    ax.set_title(f"({chr(98 + n_sec)}) Distribucion de espesor de las secciones")

fig.tight_layout()
fig.savefig(SALIDA / "fig_optimizacion_pala.png", bbox_inches="tight")
plt.show()